In [3]:
import os, numpy as np, pandas as pd

# ======== CONFIG (change per instrument) ========
IN_DIR    = r"C:\Users\loci_\Desktop\trading_webapp\DATA\all_input_files"
OUT_DIR   = r"C:\Users\loci_\Desktop\trading_webapp\DATA\all_output_files"
INSTRUMENT = "RX1_small.csv"   # e.g. "RX1_small.csv"
OUT_PREFIX = "RX1"             # e.g. "RX1"
DISTANCE   = 1/12              # AD1 monthly roll → 1/12 ; RX1 quarterly → 3/12

TRADING_DAYS = 256
CAP = 20.0
LOOKBACK_N = 36
ALPHA = 2 / (LOOKBACK_N + 1)   # std-dev decay
FORECAST_SCALER = 30.0         # carry scaler

os.makedirs(OUT_DIR, exist_ok=True)

# ======== LOAD ========
path = os.path.join(IN_DIR, INSTRUMENT)
df = pd.read_csv(path)
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

# robust column names
def find_col(d, cands):
    m = {c.lower(): c for c in d.columns}
    for c in cands:
        if c.lower() in m: return m[c.lower()]
    raise KeyError(f"Missing columns; tried {cands}, found {list(d.columns)}")

near_col = find_col(df, ["near"])
far_col  = find_col(df, ["far"])
px_col   = find_col(df, ["PX_CLOSE_1D", "px_close_1d", "Close", "close"])

df["near"] = df[near_col].astype(float)
df["far"]  = df[far_col].astype(float)
px = df[px_col].astype(float)

# ======== CARRY per your spec ========
# return = Δ near ; square_returns = return^2
df["return"] = df["near"].diff()
sq = df["return"]**2

# EWMA variance on squared returns; first = first sq
var = np.full(len(df), np.nan)
arr = sq.to_numpy()
i0 = np.argmax(~np.isnan(arr)) if np.any(~np.isnan(arr)) else None
if i0 is not None:
    var[i0] = arr[i0]
    prev = var[i0]
    for i in range(i0+1, len(arr)):
        x2 = 0.0 if np.isnan(arr[i]) else arr[i]
        prev = ALPHA * x2 + (1 - ALPHA) * prev
        var[i] = prev

ann_std = pd.Series(np.sqrt(var), index=df.index) * np.sqrt(TRADING_DAYS)

# price_difference = far - near ; net_expected_return = price_difference / DISTANCE
df["price_difference"]   = df["far"] - df["near"]
df["net_expected_return"] = df["price_difference"] / float(DISTANCE)

# raw_carry = net_expected_return / annual_standard_deviation
den = ann_std.replace(0.0, np.nan)
raw_carry = df["net_expected_return"] / den

# forecast (scaled & capped)
df["forecast_carry"] = (raw_carry * FORECAST_SCALER).clip(-CAP, CAP)

# ======== RAW PnL proxy & Sharpe (today forecast × tomorrow price_difference) ========
raw_pnl = df["forecast_carry"] * df["price_difference"].shift(-1)
mu, sd = raw_pnl.mean(), raw_pnl.std()
sharpe = np.nan if (sd is None or sd == 0 or np.isnan(sd)) else (mu / sd) * np.sqrt(TRADING_DAYS)

print(f"{OUT_PREFIX} Carry raw Sharpe: {sharpe:.3f}")

# ======== SAVE ========
out = df[["Date", "near", "far", "forecast_carry"]].copy()
# keep optional diagnostics too (uncomment if you want them in CSV)
# out["price_difference"] = df["price_difference"]
# out["raw_pnl_proxy"]    = raw_pnl

out_path = os.path.join(OUT_DIR, f"{OUT_PREFIX}_CARRY.csv")
out.to_csv(out_path, index=False)
print(f"Saved: {out_path}")


RX1 Carry raw Sharpe: 25.253
Saved: C:\Users\loci_\Desktop\trading_webapp\DATA\all_output_files\RX1_CARRY.csv
